In [12]:
%load_ext autoreload
%autoreload 2
%reset -f

from locallib.picarrodb import *
from locallib.box import *
from locallib.query import *
from locallib.pandas import *

import sqlite3
import os
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Connect to the Italgas databaes

In [13]:
DATABASE_PATH = os.path.join(os.getcwd(), "../database/" "italgas_g2g_anders.db")
# Create a connection to the Italgas database using sqlite3
conn = sqlite3.connect(DATABASE_PATH)
print(DATABASE_PATH)

/home/sandbox/personal-repos/DA-3590/dump/../database/italgas_g2g_anders.db


In [14]:
query = 'SELECT * FROM LEAKS'
read_df = pd.read_sql_query(query, conn)
print(len(read_df))

239061


In [15]:
read_df['lisa'] = read_df['lisa'].str.replace('-LISA', '-L-')

In [16]:
a = Query(get_final_reports('Italgas',years =[2025]))
reports = a.execute([EU1_Conn, EU2_Conn])

In [17]:
print(len(reports))

3199


In [18]:
emission_source = f"""SELECT ES.ReportId, ES.Id, ES.IsFiltered FROM EmissionSource ES 
    WHERE ES.ReportId IN (SELECT ReportId FROM #TempReports)
    AND (ES.Disposition = 1 OR ES.Disposition =3)"""
reports.db.set_query(emission_source)
emission_source_df = reports.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReports', source_col = 'ReportId', append = False)
print(len(emission_source_df))


190919


In [23]:
emission_source_df[emission_source_df['IsFiltered'] == False].shape

(48075, 3)

In [ ]:
ri_query = f"""SELECT * FROM ReportInvestigation RI WHERE RI.BoxId IN (SELECT BoxId FROM #TempBoxes)"""
box_df.db.set_query(ri_query)
ri_df = box_df.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempBoxes', source_col = 'BoxId', append = False)
ri_df.rename(columns = {'Id': 'ReportInvestigationId'}, inplace = True)

In [24]:
box_query = f"""SELECT B.Id as BoxId, 
                B.ReportId, B.UniqueIdentifier,
                (SELECT Description FROM InvestigationStatusTypes IST WHERE IST.Id = B.InvestigationStatusTypeId) AS InvestigationStatusName
                FROM Box B 
                WHERE B.ReportId IN (SELECT ReportId FROM #TempReports)
                AND B.UniqueIdentifier NOT LIKE '%G%'"""           
           
box_df = emission_source_df.db.set_query(box_query)
box_df = box_df.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReports', source_col = 'ReportId', append = False)
print(len(box_df))

48076


In [25]:
box_df

,BoxId,ReportId,UniqueIdentifier,InvestigationStatusName
0,62798AA2-9087-4FE6-B19A-000236AF18A9,254F9529-E7D5-4B5E-D893-3A19C7BECDBD,CR-254F95-L-22,Not Investigated
1,EEE375DC-3D39-4332-87AC-00117B56B3F2,5F6C145A-1BB8-E3F1-3757-3A183B018A62,CR-5F6C14-L-67,Not Investigated
2,6820D5DA-F9DA-418B-9CCF-001BEDE45F91,6EA39CFF-946A-ECC8-D2CF-3A1946CFB201,CR-6EA39C-L-3,Not Investigated
3,D8077500-578F-48A2-8D81-0026BC43CBF9,C6E5ADCC-E72C-EF86-BACF-3A1CD67E32A0,CR-C6E5AD-L-15,Not Investigated
4,E21E9357-691B-4EAD-8D92-00315051C6D2,BA4A0EBA-47CE-C6A7-EE78-3A1D84555E5F,CR-BA4A0E-L-14,Not Investigated
...,...,...,...,...
48071,052FBE50-50C2-4511-BCC4-FFCA06A90F3F,2195F5AF-E39B-FB23-5D2F-3A1850306658,CR-2195F5-L-21,Not Investigated
48072,E4E0DEB0-A57F-42DE-B65B-FFE66C4F6369,6A87915D-AD16-08CD-B7EC-3A1BF96B49E8,CR-6A8791-L-4,Not Investigated
48073,1E12F7A7-B2CB-479C-B4C6-FFEC3CB4D78C,FCA15E0F-5AD5-7648-FC21-3A1DFC5668A6,CR-FCA15E-L-30,Found Gas Leak
48074,484C1847-285A-4AA2-B8D0-FFEDF9E5B622,30B1585D-3DFB-8FED-3574-3A19273683D5,CR-30B158-L-4,Not Investigated


In [22]:
rii_query = f""" SELECT RII.ReportInvestigationId, 
                    (SELECT CustomLabel FROM InvestigationTemplateItem ITI WHERE ITI.Id = RII.InvestigationTemplateItemId) AS CustomLabel, 
                    (SELECT Label FROM MasterInvestigationItem WHERE Id = (SELECT MasterInvestigationItemId FROM InvestigationTemplateItem ITI WHERE ITI.Id = RII.InvestigationTemplateItemId)) AS MasterLabel,
                    RII.SelectedValue FROM ReportInvestigationItem RII WHERE RII.ReportInvestigationId IN (SELECT ReportInvestigationId FROM #TempReportInvestigations)"""
ri_df.db.set_query(rii_query)
rii =ri_df.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReportInvestigations', source_col = 'ReportInvestigationId')

NameError: name 'ri_df' is not defined

In [ ]:
# Pivot ri_df by grouping on 'ReportInvestigationId' with 'CustomLabel' as columns and 'SelectedValue' as values
ri_pivot_df = rii.pivot(index='ReportInvestigationId', columns='MasterLabel', values='SelectedValue')
ri_pivot_df.reset_index(inplace=True) 
ri_pivot_df.head()